# 第35课 · 欧拉公式（Euler's formula）遇见 FFT — `e^{-2πikn/N}` 是什么，旋转因子可视化

**今日目标**：用 `cos θ + i·sin θ` 手动拼出 `euler(θ)`，再导出 `twiddle(k, n, N) = euler(-2πkn/N)`。

L37-L39 重写 FFT 时每一行都调用 `twiddle`；今天写好这两个函数，下周直接复用。

← **上一课**　[L34 · Nyquist 定理与混叠](L34_aliasing.ipynb)

> 上节课学习了 **Nyquist 定理与混叠**：6kHz 正弦波被 8kHz 采样后会变成什么。  
> 本课将探讨 **欧拉公式遇见 FFT**。

## 学习目标

完成本课后，你应该能够：

1. **实现 `euler(theta)`**：用 `np.cos` 和 `np.sin` 从第一原理构造 `e^{iθ}`，不依赖 `np.exp`。
2. **实现 `twiddle(k, n, N)`**：调用 `euler` 计算 FFT 旋转因子 `e^{-2πikn/N}`，说明负号的物理含义（顺时针 / 分析方向）。
3. **通过断言验收**：让验证 cell 的所有 assert 全部通过，包括 `twiddle(1,2,8) == -1j` 这个非平凡 case。
4. **连接 Aurora 源码**：能在 `src/aurora/audio/transforms.py` 中找到使用等效旋转因子矩阵的代码位置。

## 复习桥 · L07 方波 = 谐波叠加（2 分钟）

L07 已证：周期波形可写成正弦波的加权和——方波 ≈ 奇次谐波 `sin(2πkt)/k` 的叠加。

DFT/FFT 做的是**逆问题**：给定一段信号，问「每个频率分量有多大？」
今天写的 `twiddle(k, n, N)` 就是离散时间第 `n` 步、频率 `k` 上的旋转位置。

带着这个对照进入欧拉公式：复指数不是新东西，是把 sin/cos 合并成单位圆上的坐标。


## 本课剧情：旋转的密码

FFT 的核心公式里有个神秘指数：`e^{-2πikn/N}`。它不是普通的实数指数，而是一个**在单位圆上旋转的复数**。

**欧拉公式（Euler's formula）**：
$$e^{i\theta} = \cos\theta + i\sin\theta$$

直觉：想象一只手表的秒针，固定在单位圆上。角度 θ 告诉它转到哪个位置——横坐标是 cos θ，纵坐标是 sin θ。把这两个坐标合并成一个复数，就是 `e^{iθ}`。

**为什么 FFT 需要它？**

DFT（离散傅里叶变换）的第 k 个输出：

$$X[k] = \sum_{n=0}^{N-1} x[n] \cdot e^{-2\pi i k n/N}$$

每个 `e^{-2πikn/N}` 都是一个**旋转因子（twiddle factor）**：秒针以频率 k 旋转，在第 n 步时到达的位置。把信号 x[n] 乘以这个旋转因子，相当于"问 x[n] 在这个频率上的投影有多大"。

本节实现两个函数：
- `euler(theta)` → `cos(theta) + i·sin(theta)` （欧拉公式）
- `twiddle(k, n, N)` → `e^{-2πi·k·n/N}` （FFT 旋转因子）

## 开课前 2 分钟复习：方波 = 谐波叠加

- 你已经见过奇次谐波叠加方波的轮廓
- 现在要把这些正弦分量写成 `e^{-2πikn/N}` 的旋转因子
- 负号只是告诉你：这是分析方向，旋转会顺时针走

L07 的直觉会在这里重新出现，只不过这次用欧拉公式来描述。

## 1. 先把复数当成平面坐标

- 复数 `a + b·i` 对应平面上的点 `(a, b)`。
- 实部 `a` 是横坐标，虚部 `b` 是纵坐标。
- 乘以 `i` 会把点逆时针旋转 90°。
- `e^{iθ}` 是单位圆（unit circle）上角度为 `θ` 的点，也就是 `(cos θ, sin θ)`。

**为什么用 cos+i·sin 手动拼，而不是直接写 `np.exp(1j*theta)`？** 两种写法数值完全相同，但手动拼出实部和虚部可以让你亲眼看到欧拉公式成立——而不只是把它当成黑盒函数调用。L37-L39 重写 FFT 时，每对 `(k, n)` 调用一次 `twiddle`，N² 次调用后拼出完整频谱；今天理解底层构造，下周才能看懂每行矩阵在做什么。

## 1.1 复数乘法先看 90° 旋转

先不进入公式，只看 `z * 1j`。每乘一次 `1j`，点就绕原点逆时针转 90°。


In [ ]:
import numpy as np

z = 1 + 0j
for step in range(5):
    print(f'step={step}  z={z.real:+.0f}{z.imag:+.0f}j')
    z = z * 1j


## 实验入口：观察角度序列

这组实验用均匀分布在 [0, 2π) 的角度作为输入，同时对照 cos、sin 和复数坐标，确认三者始终指向单位圆上同一个点。

In [ ]:
import numpy as np
print('就绪；虚数单位示例:', 1j * 1j)  # = (-1+0j)

## 动手观察：从角度序列到复数坐标

调整 `N`（总采样点数）或 `k`（频率编号），观察 `twiddle` 输出的复数点如何在单位圆上重新排列。

In [ ]:
import numpy as np

sample_rate = 8
duration = 1.0
freq = 2.0
N = round(duration * sample_rate)
n = np.arange(N)
t = n / sample_rate
angle = 2 * np.pi * freq * t
wave = np.sin(angle)

print('N =', N)
print('采样点编号 n =', n)
print('时间轴 t =', np.round(t, 3))
print('角度 angle =', np.round(angle, 3))
print('sin(angle) =', np.round(wave, 3))


## 代码实验：角度、三角函数和复数坐标对齐

并排打印同一批角度的 cos、sin 和 exp(1j·θ) 三列，验证 exp(1j·θ) 的实部等于 cos θ、虚部等于 sin θ。

In [ ]:
import numpy as np

angles = np.linspace(0, 2*np.pi, 9)
for theta in angles:
    z = np.exp(1j * theta)
    print(f'theta={theta:5.2f} | cos={z.real:6.3f} | sin={z.imag:6.3f} | z={z.real:6.3f}{z.imag:+6.3f}j')


## 2. ✏️ 实现 `euler(theta)`

**手算三个特殊角度**（单位圆上的关键点）：

| theta (弧度) | cos θ | sin θ | e^{iθ} |
|---|---|---|---|
| 0 | 1 | 0 | **1+0j** |
| π/2 ≈ 1.5708 | 0 | 1 | **0+1j**（逆时针转90°） |
| π ≈ 3.1416 | -1 | 0 | **-1+0j**（转180°，秒针指左） |
| 3π/2 | 0 | -1 | **0-1j** |
| 2π | 1 | 0 | **1+0j**（转一圈，回原点） |

**关键性质**：|e^{iθ}| = √(cos²θ + sin²θ) = **1**（永远在单位圆上）

> **实现提示**：`return np.cos(theta) + 1j * np.sin(theta)`

### 写代码前，先把变量表补完整

写 `euler` 前明确三件事：
- 输入：`theta`（弧度，标量或数组）
- 关键步骤：计算 `np.cos(theta)` 作实部，`np.sin(theta)` 作虚部，相加得复数
- 返回：复数（或复数数组），模为 1，辐角为 theta

In [ ]:
def euler(theta):
    # ✏️ TODO: 返回 cosθ + i·sinθ
    raise NotImplementedError("TODO: 实现 euler(theta) — 返回 cos(theta) + 1j*sin(theta)")


## 3. 从单位圆到 FFT 旋转因子

`e^{iθ}` 表示“转到角度 θ 的位置”。FFT 需要的是一组规则旋转：

- `N`：一圈分成多少份
- `n`：当前采样点编号
- `k`：当前频率编号
- `-2π·k·n/N`：这个频率在这个采样点要转到的角度

所以 `twiddle(k, n, N)` 只是把这个角度送进欧拉公式。


## 3. ✏️ 任务二：FFT 旋转因子 `twiddle`

`W_N^{kn} = e^{-2πi·k·n/N}`

**推理路线**：
1. 旋转因子的角度是 `-2π·k·n/N`；负号是 DFT 的习惯约定，表示顺时针旋转（分析频率分量时向右转）。
2. 把这个角度送入 `euler(theta)` 即可，不需要重新推导 cos/sin。
3. 优先调用 `euler` 而不是 `np.exp`：若要替换底层实现，改一处 `euler` 即可全局生效。

**参考输入输出**：`twiddle(k=1, n=2, N=8)` = `euler(-π/2)` = 0-1j

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


In [ ]:
def twiddle(k, n, N):
    # ✏️ TODO: 返回 e^{-2πi·k·n/N}
    raise NotImplementedError("TODO: 实现 twiddle(k, n, N) — 返回 euler(-2*np.pi*k*n/N)")


## 4. 检查 + 打印单位圆上的点

In [ ]:
theta = np.linspace(0, 2*np.pi, 9)   # 0..360°, 9 个点
z = euler(theta)
assert np.max(np.abs(z - np.exp(1j*theta))) < 1e-12, 'euler 应等于 np.exp(1j*theta)'
assert np.allclose(np.abs(z), 1.0), 'e^{iθ} 模长恒为 1（在单位圆上）'
assert abs(twiddle(0, 0, 8) - 1) < 1e-12, 'k·n=0 时旋转因子应为 1'
assert abs(twiddle(1, 2, 8) - (0-1j)) < 1e-12, 'twiddle(1,2,8) 应为 -1j（非平凡验证）'
for ang, val in zip(np.degrees(theta), z):
    print(f'{ang:6.0f}° | {val.real:+.3f} {val.imag:+.3f}i')
print('\n✅ 复指数=旋转，twiddle=FFT 里的旋转因子。')

## 5. 画出单位圆

In [ ]:
import matplotlib.pyplot as plt
fine = euler(np.linspace(0, 2*np.pi, 200))
plt.figure(figsize=(4.5, 4.5))
plt.plot(fine.real, fine.imag, '-', lw=1)
plt.plot(z.real, z.imag, 'o')
plt.gca().set_aspect('equal')
plt.title('e^{iθ} traces the unit circle')
plt.xlabel('Re'); plt.ylabel('Im'); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

**🎉 完成后**：`git commit -m 'learn: L35 euler'`（L37-L39 重写 FFT 全靠今天）

## 🎨 图示：8 个旋转因子 = FFT 的积木

In [ ]:
import aurora.aviz as aviz; aviz.style()
aviz.twiddles(8);

In [ ]:
angles = np.linspace(0, 2*np.pi, 5)
for theta in angles:
    z = np.exp(1j*theta)
    reconstructed = np.cos(theta) + 1j*np.sin(theta)
    print(f'theta={theta:.2f} | exp={z:.2f} | cos+i sin={reconstructed:.2f} | match={np.allclose(z, reconstructed)}')


## 参数实验：只改一个旋钮

计算 `np.array([twiddle(1, n, 8) for n in range(8)])` 得到 8 个旋转因子，打印它们的相位（角度），确认分布为 0°, -45°, -90°, -135°, 180°, 135°, 90°, 45°——八个点均匀排列在单位圆上。

把 `N` 从 8 改成 16：相邻旋转因子的角度间距缩小一半（-22.5°），单位圆上的点密度翻倍。把 `k` 从 1 改成 3：每步旋转角度变为 -135°，FFT 频率编号直接控制旋转速度。

In [ ]:
# ✏️ 参数实验：只改一个旋钮
import numpy as np

# --- 实验 A：固定 k=1，N=8，遍历 n=0..7 ---
N = 8
factors = np.array([twiddle(1, n, N) for n in range(N)])
phases_deg = np.degrees(np.angle(factors))
print('N=8, k=1 时各旋转因子的相位（度）：')
for n, (f, p) in enumerate(zip(factors, phases_deg)):
    print(f'  n={n}: {f.real:+.3f}{f.imag:+.3f}j  ∠{p:+.1f}°')

# --- 实验 B：改 N=16，观察角度间距缩小一半 ---
N2 = 16
factors2 = np.array([twiddle(1, n, N2) for n in range(N2)])
phases2 = np.degrees(np.angle(factors2))
print(f'\nN={N2}, k=1 时相邻旋转因子角度间距：{abs(phases2[1]-phases2[0]):.1f}°（N=8 时为 45°，缩小了一半）')

## 本课收束

现在可以调用 `euler(theta)` 生成单位圆上任意角度的复数，也可以调用 `twiddle(k, n, N)` 计算 FFT 所需的旋转因子。这两个函数对应 `src/aurora/audio/transforms.py` 里 L37-L39 DFT 实现的底层调用。下一课（L36）会用窗函数对信号加权，处理后的信号最终会送进今天写好的 `twiddle`。

本节与 `notebooks/1_complex_trig/L06_euler.ipynb` 覆盖同一个公式，侧重点不同：x3 建立欧拉公式的数学直觉（单位圆图示、复平面几何），本节把它落地为 FFT 旋转因子的实现。先做过 x3 的话今天的推导会更流畅；没做过的话，完成本节后再回去看 x3 的单位圆图示，两边会相互印证。

In [ ]:
# 小检查：从短序列开始，确认每一步输出
import numpy as np

sample_rate = 8
duration = 1.0
freq = 1.0
N = round(duration * sample_rate)
n = np.arange(N)
t = n / sample_rate
angle = 2 * np.pi * freq * t
x = np.sin(angle)

print('1) N 应该是多少？', N)
print('2) n 是采样点编号：', n)
print('3) t 是每个点的秒数：', np.round(t, 3))
print('4) angle 是每个点在圆上的角度：', np.round(angle, 3))
print('5) x 是最终波形：', np.round(x, 3))


---
⬇️ **通关检验**：收束小结已读；请完成下方白板挑战后再勾选自评。


## ✏️ 白板挑战：欧拉公式与旋转因子手算（目标 10 分钟）

盖上屏幕，纸上作答：

**问 1**：手算 `euler(θ)` 的三个特殊值（无需计算器）：
- euler(0) = ?
- euler(π/2) = ?
- euler(π) = ?

**问 2**：验证模长性质：`|euler(θ)|` 对任意 θ 都等于 ?  
（用勾股定理：√(cos²θ + sin²θ) = ?）

**问 3**：旋转因子 `twiddle(k=1, n=2, N=8)` = ?  
- 角度 = -2π·1·2/8 = ?（弧度）
- cos(?) + i·sin(?) = ?（精确值）

**问 4**：`twiddle(k=0, n=n, N=N)` 对任意 n、N 的值是 ?（为什么？）

**问 5**：N=8 时，`twiddle(1, n, 8)` 对 n=0,1,2,...,7 在单位圆上的排列规律是什么？

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：euler 特殊值
try:
    e0   = euler(0)
    e90  = euler(np.pi/2)
    e180 = euler(np.pi)
    assert np.isclose(e0,   1+0j,  atol=1e-12), f"euler(0) 应=1+0j，得到 {e0}"
    assert np.isclose(e90,  0+1j,  atol=1e-12), f"euler(π/2) 应=0+1j，得到 {e90}"
    assert np.isclose(e180, -1+0j, atol=1e-12), f"euler(π) 应=-1+0j，得到 {e180}"
    print(f"Q1 ✅  euler(0)={e0}  euler(π/2)={e90}  euler(π)={e180}")
except NotImplementedError:
    print("⬜ Q1：请先实现 euler()，再运行对答案格")

# 问2：模长恒=1
thetas = np.linspace(0, 2*np.pi, 1000)
mags = np.abs(np.cos(thetas) + 1j*np.sin(thetas))
assert np.allclose(mags, 1.0, atol=1e-12)
print(f"Q2 ✅  |euler(θ)| = 1.0 对所有 θ 成立（单位圆性质）")

# 问3：twiddle(1, 2, 8) = e^{-πi/2} = 0 - 1j
angle_312 = -2 * np.pi * 1 * 2 / 8  # = -π/2
expected_312 = np.cos(angle_312) + 1j*np.sin(angle_312)
assert np.isclose(expected_312, 0-1j, atol=1e-12), f"twiddle(1,2,8) 应=0-1j，得到 {expected_312}"
try:
    tw = twiddle(1, 2, 8)
    assert np.isclose(tw, 0-1j, atol=1e-12)
    print(f"Q3 ✅  twiddle(1,2,8) = e^{{-πi/2}} = {tw}")
except NotImplementedError:
    print("⬜ Q3：请先实现 twiddle()，再运行对答案格")

# 问4：twiddle(0, n, N) = e^0 = 1 对所有 n
try:
    for n_val in range(8):
        assert np.isclose(twiddle(0, n_val, 8), 1+0j, atol=1e-12)
    print(f"Q4 ✅  twiddle(0,n,N)=e^0=1 对所有 n（k=0 对应 DC 分量，旋转因子为1）")
except NotImplementedError:
    print("⬜ Q4：请先实现 twiddle()，再运行对答案格")

# 问5：N=8 的旋转因子均匀分布在单位圆
angles_8 = np.array([-2*np.pi*1*n/8 for n in range(8)])
tw_8 = np.cos(angles_8) + 1j*np.sin(angles_8)
assert np.allclose(np.abs(tw_8), 1.0, atol=1e-12)  # 都在单位圆
# 相邻角差固定 = -π/4
diffs = np.diff(np.angle(tw_8))
assert np.allclose(diffs, diffs[0], atol=1e-12)
print(f"Q5 ✅  8个旋转因子均匀分布，每步转 -π/4={np.degrees(-np.pi/4):.1f}°")
print("\n🎉 欧拉公式白板挑战通过！旋转因子 e^{{-2πikn/N}} 是 FFT 的核心积木。")

In [ ]:
# ✏️ 本课自评
l35_review = {
    "euler_formula":          None,  # 记住 e^{iθ}=cosθ+i·sinθ？True/False
    "euler_implemented":      None,  # euler(theta) 实现并通过断言？True/False
    "twiddle_implemented":    None,  # twiddle(k,n,N) 实现并通过断言？True/False
    "unit_circle_property":   None,  # 理解 |e^{iθ}|=1（单位圆）？True/False
    "whiteboard_passed":      None,  # 白板挑战纸上推导完成？True/False
}

unfilled = [k for k, v in l35_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l35_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L35 全部通关！进入 L36：窗函数原理')

---

→ **下一课**　[L36 · 窗函数原理](L36_windows.ipynb)

> 下节课将学习 **窗函数原理**：矩形窗的旁瓣泄漏，Hann / Hamming / Blackman 对比。